<a href="https://colab.research.google.com/github/bavucadoz/244107020162_ML_2026/blob/main/Minggu06/P04_Week06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 4**
*Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.*


---



In [ ]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520039 sha256=4ffab7a1756fbb59c95cf9f13418a1cc198783a270e856af15b8cc0db707fcb1
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679795 sha256=663f617746a67885f357d90f56d0b92244e589a67a962b8463bb139a15a859a7
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 40.01617741584778 detik
Query time: 0.0009965896606445312 detik
Neighbors: [28539, 235071, 321353, 559484, 469147] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.04147815704345703 detik
Query time: 0.01922154426574707 detik
Neighbors: [ 28539 235071 321353 559484 469147] ...

=== HNSW (hnswlib) ===
Build time: 215.07839393615723 detik
Query time: 0.00042128562927246094 detik
Neighbors: [ 28539 235071 321353 559484 469147] ...




---

**Tabel Percobaan**

In [ ]:
import numpy as np
import time
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

def run_comparison(metric, n_data, dim):
    np.random.seed(42)
    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)
    k = 5

    results = []

    # 1. Annoy
    ann_metric = 'euclidean' if metric == 'Euclidean' else 'angular'
    ann_index = AnnoyIndex(dim, ann_metric)

    start = time.perf_counter()
    for i in range(n_data):
        ann_index.add_item(i, X[i])
    ann_index.build(10) # 10 trees
    build_ann = time.perf_counter() - start

    start = time.perf_counter()
    idx_ann = ann_index.get_nns_by_vector(query[0], k)
    query_ann = (time.perf_counter() - start) * 1000 # dalam ms

    results.append({
        "Metrik": metric, "Dimensi": f"{dim}D", "Jumlah Data": f"{n_data:,}",
        "Algoritma": "Annoy", "Index Terdekat": f"[{' '.join(map(str, idx_ann))}]",
        "Waktu Build (s)": f"{build_ann:.4f}", "Waktu Query (ms)": f"{query_ann:.4f}"
    })

    # 2. FAISS (IndexFlat)
    # Copy array agar tidak merusak data asli saat dinormalisasi untuk Cosine
    X_faiss = X.copy()
    query_faiss = query.copy()

    start = time.perf_counter()
    if metric == 'Euclidean':
        faiss_index = faiss.IndexFlatL2(dim)
    else:
        # FAISS Cosine = Inner Product pada vektor yang dinormalisasi L2
        faiss.normalize_L2(X_faiss)
        faiss.normalize_L2(query_faiss)
        faiss_index = faiss.IndexFlatIP(dim)

    faiss_index.add(X_faiss)
    build_faiss = time.perf_counter() - start

    start = time.perf_counter()
    _, idx_faiss = faiss_index.search(query_faiss, k)
    query_faiss = (time.perf_counter() - start) * 1000 # dalam ms

    results.append({
        "Metrik": metric, "Dimensi": f"{dim}D", "Jumlah Data": f"{n_data:,}",
        "Algoritma": "FAISS", "Index Terdekat": f"[{' '.join(map(str, idx_faiss[0]))}]",
        "Waktu Build (s)": f"{build_faiss:.4f}", "Waktu Query (ms)": f"{query_faiss:.4f}"
    })

    # 3. HNSW (hnswlib)
    hnsw_metric = 'l2' if metric == 'Euclidean' else 'cosine'
    hnsw_index = hnswlib.Index(space=hnsw_metric, dim=dim)

    start = time.perf_counter()
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw_index.add_items(X)
    build_hnsw = time.perf_counter() - start

    hnsw_index.set_ef(50)

    start = time.perf_counter()
    idx_hnsw, _ = hnsw_index.knn_query(query, k=k)
    query_hnsw = (time.perf_counter() - start) * 1000 # dalam ms

    results.append({
        "Metrik": metric, "Dimensi": f"{dim}D", "Jumlah Data": f"{n_data:,}",
        "Algoritma": "HNSW", "Index Terdekat": f"[{' '.join(map(str, idx_hnsw[0]))}]",
        "Waktu Build (s)": f"{build_hnsw:.4f}", "Waktu Query (ms)": f"{query_hnsw:.4f}"
    })

    return results

scenarios = [
    ('Euclidean', 1000, 2),
    ('Euclidean', 1000, 5),
    ('Euclidean', 1000000, 2),
    ('Euclidean', 1000000, 5),
    ('Cosine', 1000, 2),
    ('Cosine', 1000, 5),
    ('Cosine', 1000000, 2),
    ('Cosine', 1000000, 5)
]

all_results = []
for m, n, d in scenarios:
    all_results.extend(run_comparison(m, n, d))

df = pd.DataFrame(all_results)
print("\n")
print(df.to_markdown(index=False, stralign='left'))



| Metrik    | Dimensi   | Jumlah Data   | Algoritma   | Index Terdekat                       |   Waktu Build (s) |   Waktu Query (ms) |
|:----------|:----------|:--------------|:------------|:-------------------------------------|------------------:|-------------------:|
| Euclidean | 2D        | 1,000         | Annoy       | [112 535 777 246 763]                |            0.0423 |             0.086  |
| Euclidean | 2D        | 1,000         | FAISS       | [112 535 777 246 763]                |            0.0002 |             0.1154 |
| Euclidean | 2D        | 1,000         | HNSW        | [112 535 777 246 763]                |            0.1042 |             0.1014 |
| Euclidean | 5D        | 1,000         | Annoy       | [988 780 27 943 93]                  |            0.0293 |             0.0736 |
| Euclidean | 5D        | 1,000         | FAISS       | [988 780 27 943 93]                  |            0.0001 |             0.0617 |
| Euclidean | 5D        | 1,000         | HNSW